In [0]:
# Create gold schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS 03_gold_catalog.gold_schema")
print("✓ Gold schema created/verified\n")

print("=" * 80)
print("GOLD LAYER - STAR SCHEMA FOR RETAIL ANALYTICS (TECHNOVA)")
print("=" * 80)
print("\nBuilding Fact & Dimension tables for KPI analysis")
print("Architecture: Star Schema (Fact tables + Dimension tables)\n")

In [0]:
print("\n" + "=" * 80)
print("STEP 1: CREATING DIMENSION TABLES")
print("=" * 80)

# DIM_PRODUCT - Product dimension
print("\n1. Creating dim_product...")
spark.sql("""
CREATE OR REPLACE TABLE 03_gold_catalog.gold_schema.dim_product AS
SELECT 
    productid AS product_key,
    itemnamedescription AS product_name,
    categorytype AS product_category,
    CAST(basecost AS DECIMAL(10,2)) AS unit_cost,
    CAST(markedprice AS DECIMAL(10,2)) AS unit_price
FROM 02_silver_catalog.silver_schema.mst_product_master_clean
""")
print("✓ dim_product created")

# DIM_CUSTOMER - Customer dimension
print("\n2. Creating dim_customer...")
spark.sql("""
CREATE OR REPLACE TABLE 03_gold_catalog.gold_schema.dim_customer AS
SELECT 
    custrefid AS customer_key,
    fullname AS customer_name,
    contactinfo AS contact_info,
    CASE 
        WHEN UPPER(gendercode) IN ('M', 'MALE') THEN 'Male'
        WHEN UPPER(gendercode) IN ('F', 'FEMALE') THEN 'Female'
        ELSE 'Unknown'
    END AS gender,
    COALESCE(
        TRY_TO_DATE(joineddate, 'dd-MM-yyyy'),
        TRY_TO_DATE(joineddate, 'MM-dd-yyyy'),
        TRY_TO_DATE(joineddate, 'MMM dd, yyyy'),
        TRY_TO_DATE(joineddate, 'yyyy.MM.dd'),
        TRY_TO_DATE(joineddate, 'dd-MMM-yy'),
        TRY_TO_DATE(joineddate, 'yyyy-MM-dd')
    ) AS join_date
FROM 02_silver_catalog.silver_schema.crm_cust_base_clean
""")
print("✓ dim_customer created")

# DIM_STORE - Store dimension
print("\n3. Creating dim_store...")
spark.sql("""
CREATE OR REPLACE TABLE 03_gold_catalog.gold_schema.dim_store AS
SELECT 
    stid AS store_key,
    locationnameaddress AS store_name,
    cityregion AS store_region,
    mgrcontact AS manager_contact
FROM 02_silver_catalog.silver_schema.stores_geo_list_final_clean
""")
print("✓ dim_store created")

print("\n✅ All dimension tables created successfully!")

In [0]:
print("\n" + "=" * 80)
print("STEP 2: CREATING FACT_SALES (Main Transactional Fact)")
print("=" * 80)

spark.sql("""
CREATE OR REPLACE TABLE 03_gold_catalog.gold_schema.fact_sales AS
SELECT 
    s.trxnid AS transaction_key,
    s.custid99 AS customer_key,
    s.prodcodeid AS product_key,
    s.storelocid AS store_key,
    
    -- Parse transaction date using COALESCE to try multiple formats
    COALESCE(
        TRY_TO_DATE(s.dateref, 'dd-MM-yyyy'),
        TRY_TO_DATE(s.dateref, 'MM-dd-yyyy'),
        TRY_TO_DATE(s.dateref, 'MMM dd, yyyy'),
        TRY_TO_DATE(s.dateref, 'yyyy.MM.dd'),
        TRY_TO_DATE(s.dateref, 'dd-MMM-yy'),
        TRY_TO_DATE(s.dateref, 'yyyy-MM-dd')
    ) AS transaction_date,
    
    -- Measures
    CAST(s.qtysold AS INT) AS quantity_sold,
    CAST(REGEXP_REPLACE(s.unitprice, '[^0-9.]', '') AS DECIMAL(10,2)) AS unit_price,
    
    -- Calculate total sales amount (qty * price)
    CAST(s.qtysold AS INT) * CAST(REGEXP_REPLACE(s.unitprice, '[^0-9.]', '') AS DECIMAL(10,2)) AS total_sales_amount,
    
    -- Calculate discount (marked price - actual selling price)
    (p.unit_price - CAST(REGEXP_REPLACE(s.unitprice, '[^0-9.]', '') AS DECIMAL(10,2))) * CAST(s.qtysold AS INT) AS discount_amount,
    
    -- Calculate profit (sales - cost)
    (CAST(REGEXP_REPLACE(s.unitprice, '[^0-9.]', '') AS DECIMAL(10,2)) - p.unit_cost) * CAST(s.qtysold AS INT) AS profit_amount,
    
    -- Product details for analysis
    p.product_name,
    p.product_category
    
FROM 02_silver_catalog.silver_schema.raw_sales_dtl_clean s
LEFT JOIN 03_gold_catalog.gold_schema.dim_product p 
    ON s.prodcodeid = p.product_key
WHERE s.trxnid IS NOT NULL
""")

print("✓ fact_sales created")
print("\nThis table contains:")
print("  - Transaction-level sales data")
print("  - Foreign keys to dimensions (customer, product, store)")
print("  - Measures: quantity, price, discount, profit")
print("  - Calculated fields: total_sales_amount, discount_amount, profit_amount")
print("\n✅ Fact table created successfully!")

In [0]:
print("\n" + "=" * 80)
print("STEP 3: CREATING FACT_RETURNS")
print("=" * 80)

spark.sql("""
CREATE OR REPLACE TABLE 03_gold_catalog.gold_schema.fact_returns AS
SELECT 
    r.rtnidno AS return_key,
    r.orgnltrxnid AS transaction_key,
    
    -- Get product and customer from original sale
    fs.product_key,
    fs.customer_key,
    fs.store_key,
    
    -- Parse return date using COALESCE to try multiple formats
    COALESCE(
        TRY_TO_DATE(r.rtndatestring, 'dd-MM-yyyy'),
        TRY_TO_DATE(r.rtndatestring, 'MM-dd-yyyy'),
        TRY_TO_DATE(r.rtndatestring, 'MMM dd, yyyy'),
        TRY_TO_DATE(r.rtndatestring, 'yyyy.MM.dd'),
        TRY_TO_DATE(r.rtndatestring, 'dd-MMM-yy'),
        TRY_TO_DATE(r.rtndatestring, 'dd/MM/yyyy'),
        TRY_TO_DATE(r.rtndatestring, 'MM/dd/yyyy'),
        TRY_TO_DATE(r.rtndatestring, 'yyyy-MM-dd')
    ) AS return_date,
    
    -- Return reason
    r.rsncode AS return_reason,
    
    -- Measures from original sale
    fs.quantity_sold AS return_quantity,
    fs.total_sales_amount AS returned_amount,
    fs.profit_amount AS lost_profit
    
FROM 02_silver_catalog.silver_schema.rtn_trans_logs_clean r
INNER JOIN 03_gold_catalog.gold_schema.fact_sales fs
    ON r.orgnltrxnid = fs.transaction_key
""")

print("✓ fact_returns created")
print("\nThis table contains:")
print("  - Return transaction data")
print("  - Links back to original sales via transaction_key")
print("  - Return quantity and returned amounts from original sale")
print("  - Return reasons for analysis")
print("\n✅ Fact returns table created successfully!")

In [0]:
print("\n" + "=" * 80)
print("STEP 4: CREATING FACT_INVENTORY")
print("=" * 80)

spark.sql("""
CREATE OR REPLACE TABLE 03_gold_catalog.gold_schema.fact_inventory AS
SELECT 
    i.skuid AS product_key,
    i.stidref AS store_key,
    
    -- Parse audit date from integer format (20250101 -> 2025-01-01)
    TO_DATE(CAST(i.lastauditdt AS STRING), 'yyyyMMdd') AS snapshot_date,
    
    -- Inventory measures
    CAST(i.stockonhand AS INT) AS stock_quantity,
    
    -- Flag out of stock items
    CASE WHEN CAST(i.stockonhand AS INT) = 0 THEN 1 ELSE 0 END AS is_out_of_stock,
    
    -- Join with product for additional context
    p.product_name,
    p.product_category,
    p.unit_cost,
    p.unit_price,
    
    -- Calculate inventory value
    CAST(i.stockonhand AS INT) * p.unit_cost AS inventory_value_cost,
    CAST(i.stockonhand AS INT) * p.unit_price AS inventory_value_retail
    
FROM 02_silver_catalog.silver_schema.inv_levels_clean i
LEFT JOIN 03_gold_catalog.gold_schema.dim_product p 
    ON i.skuid = p.product_key
""")

print("✓ fact_inventory created")
print("\nThis table contains:")
print("  - Daily inventory snapshots by product and store")
print("  - Stock quantities and out-of-stock flags")
print("  - Inventory valuation at cost and retail prices")
print("\n✅ Fact inventory table created successfully!")

In [0]:
print("\n" + "=" * 80)
print("GOLD LAYER - STAR SCHEMA SUMMARY")
print("=" * 80)

# List all gold tables
gold_tables = spark.sql("SHOW TABLES IN 03_gold_catalog.gold_schema").collect()

dimension_tables = []
fact_tables = []

for table in gold_tables:
    table_name = table.tableName
    if table_name.startswith('dim_'):
        dimension_tables.append(table_name)
    elif table_name.startswith('fact_'):
        fact_tables.append(table_name)

print("\n📊 DIMENSION TABLES:")
for dim in sorted(dimension_tables):
    df = spark.read.table(f"03_gold_catalog.gold_schema.{dim}")
    print(f"  ✓ {dim:<25} | Rows: {df.count():>6} | Columns: {len(df.columns):>2}")

print("\n📈 FACT TABLES:")
for fact in sorted(fact_tables):
    df = spark.read.table(f"03_gold_catalog.gold_schema.{fact}")
    print(f"  ✓ {fact:<25} | Rows: {df.count():>6} | Columns: {len(df.columns):>2}")

print("\n" + "=" * 80)
print("✅ STAR SCHEMA CREATED SUCCESSFULLY!")
print("=" * 80)
print("\nReady for KPI calculations and BI reporting")

In [0]:
# print("\n" + "=" * 80)
# print("SAMPLE DATA FROM STAR SCHEMA")
# print("=" * 80)

# print("\n📊 Sample from fact_sales:")
# display(spark.read.table("03_gold_catalog.gold_schema.fact_sales").limit(10))

# print("\n📊 Sample from dim_product:")
# display(spark.read.table("03_gold_catalog.gold_schema.dim_product"))

# print("\n📊 Sample from dim_customer:")
# display(spark.read.table("03_gold_catalog.gold_schema.dim_customer"))

# print("\n📊 Sample from dim_store:")
# display(spark.read.table("03_gold_catalog.gold_schema.dim_store"))